# 🗺️ Fas 1c: Geokodning — Avstånd till centrum, skolor & stationer

**Syfte:** Beräkna koordinater för varje bostad och avstånd till viktiga platser  
**Metod:** Geokoda områden via OpenStreetMap (Nominatim) — gratis, ingen API-nyckel  
**Tid:** (vi geokodar 40 områden, inte 6600 adresser)

In [ ]:
#Geokodning — Avstånd till centrum, skolor & stationer
# ============================================================
# STEG 1: Installera och importera
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'geopy'], capture_output=True)

import pandas as pd
import numpy as np
import re
import time
import json
import os
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
from geopy.extra.rate_limiter import RateLimiter

print('Paket laddade!')

Paket laddade!


In [2]:
# ============================================================
# STEG 2: Ladda data och extrahera adresser
# ============================================================
df = pd.read_csv('../data/processed/orebro_housing_enriched.csv')
print(f'Laddad: {len(df)} rader')

# Extrahera gatuadress från raw_text
# raw_text börjar med: "Såld 13 mar. 2026 Kaserngården 14 Lägenhet Rynninge..."
def extract_address(raw_text):
    if pd.isna(raw_text):
        return None
    # Mönster: efter datumet, före "Lägenhet/Villa/Radhus"
    match = re.search(
        r'(?:Såld.*?\d{4})\s+(.+?)\s+(?:Lägenhet|Villa|Radhus|Bostadsrätt|Fritidshus)',
        str(raw_text)
    )
    if match:
        return match.group(1).strip()
    return None

if 'raw_text' in df.columns:
    df['gatuadress'] = df['raw_text'].apply(extract_address)
    print(f'Extraherade adresser: {df["gatuadress"].notna().sum()} av {len(df)}')
    print(f'\nExempel:')
    for addr in df['gatuadress'].dropna().head(5):
        print(f'  {addr}')
else:
    print('Ingen raw_text-kolumn — vi använder omrade_clean istället')

Laddad: 6600 rader
Extraherade adresser: 6592 av 6600

Exempel:
  Kaserngården 14
  Lövskogsvägen 17
  Förmansgatan 8
  Hagmarksgatan 11B
  Mandelstensvägen 95


In [9]:
# ============================================================
# STEG 3: Definiera viktiga platser i Örebro
# ============================================================

# Örebro Stortorget (centrum)
CENTRUM = (59.2753, 15.2134)

# Örebro centralstation
CENTRALSTATION = (59.2773, 15.2066)

# Universitetssjukhuset Örebro (USÖ)
SJUKHUSET = (59.2722, 15.1758)

# Örebro universitet
UNIVERSITETET = (59.2537, 15.2472)

# Marieberg köpcentrum
MARIEBERG = (59.2878, 15.2517)

print('Viktiga platser definierade:')
platser = {
    'centrum': CENTRUM,
    'centralstation': CENTRALSTATION,
    'sjukhuset': SJUKHUSET,
    'universitetet': UNIVERSITETET,
    'marieberg': MARIEBERG,
}
for namn, coord in platser.items():
    print(f'  {namn}: {coord}')

Viktiga platser definierade:
  centrum: (59.2753, 15.2134)
  centralstation: (59.2773, 15.2066)
  sjukhuset: (59.2722, 15.1758)
  universitetet: (59.2537, 15.2472)
  marieberg: (59.2878, 15.2517)


In [10]:
# ============================================================
# STEG 4: Geokoda alla unika områden
# ============================================================
# Geokoda bara de 40 områdesgrupperna


# Använd samma top 40 som modellen
top_areas = df['omrade_clean'].value_counts().head(40).index.tolist()
areas_to_geocode = top_areas + ['Örebro']  # + fallback

print(f'Områden att geokoda: {len(areas_to_geocode)}')

geolocator = Nominatim(user_agent='orebro_housing_ml_v2', timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

area_coords = {}
failed = []

for omrade in areas_to_geocode:
    query = f'{omrade}, Örebro, Sverige'
    try:
        location = geocode(query)
        if location:
            area_coords[omrade] = (location.latitude, location.longitude)
            print(f'  ✓ {omrade}: ({location.latitude:.4f}, {location.longitude:.4f})')
        else:
            failed.append(omrade)
            print(f'  ✗ {omrade}: ej hittad')
    except Exception as e:
        failed.append(omrade)
        print(f'  ✗ {omrade}: {str(e)[:50]}')

print(f'\nGeokodade: {len(area_coords)} av {len(areas_to_geocode)}')
print(f'Misslyckade: {len(failed)}')

Områden att geokoda: 41
  ✓ Örebro: (59.2747, 15.2151)
  ✓ Adolfsberg: (59.2390, 15.1835)
  ✓ Almby: (59.2606, 15.2506)
  ✓ Sörbyängen: (59.2536, 15.2266)
  ✓ Hovsta: (59.3467, 15.2321)
  ✓ Mellringe: (59.2980, 15.1625)
  ✓ Lundby: (59.3089, 15.1843)
  ✓ Sörby: (59.2603, 15.2252)
  ✓ Solhaga: (59.2732, 15.1583)
  ✓ Nya Hjärsta: (59.2918, 15.1794)
  ✓ Glanshammar: (59.3183, 15.4008)
  ✓ Lillån: (59.3167, 15.2375)
  ✓ Öster: (59.2594, 15.2424)
  ✓ Stora Mellösa: (59.2106, 15.4976)
  ✓ Vintrosa: (59.2554, 14.9619)
  ✓ Odensbacken: (59.1625, 15.5255)
  ✓ Ormesta: (59.2564, 15.2727)
  ✓ Garphyttan: (59.3022, 14.9495)
  ✗ Radhus Lillån: ej hittad
  ✓ Mosås: (59.1963, 15.1540)
  ✓ Ladugårdsängen: (59.2543, 15.2044)
  ✓ Björkhaga: (59.2860, 15.1504)
  ✗ Centralt: ej hittad
  ✓ Väster: (59.4342, 15.0240)
  ✗ Centralt Öster: ej hittad
  ✗ Radhus Ormesta: ej hittad
  ✓ Marieberg: (59.2238, 15.1585)
  ✗ Lägenhet Rynningeåsen: ej hittad
  ✗ Södra Lindhult: ej hittad
  ✗ Lägenhet Örebro: ej hittad
 

In [11]:
# ============================================================
# STEG 5: Fixa misslyckade områden manuellt
# ============================================================
#
# Områden som Nominatim inte hittade sätts till centrum som fallback,
# eller manuella koordinater om vi vet var de ligger.

# Manuella koordinater för vanliga Örebro-områden som kan misslyckas
manuella = {
    'Örebro': CENTRUM,
    'Centralt': CENTRUM,
    'Centralt Öster': (59.2753, 15.2200),
    'Centralt Väster': (59.2753, 15.2050),
    'Öster': (59.2753, 15.2300),
    'Väster': (59.2753, 15.1950),
    'Söder': (59.2650, 15.2134),
    'Norr': (59.2850, 15.2134),
}

for omrade in failed:
    if omrade in manuella:
        area_coords[omrade] = manuella[omrade]
    else:
        # Fallback till centrum
        area_coords[omrade] = CENTRUM
        print(f'  {omrade} → fallback till centrum')

print(f'Totalt geokodade: {len(area_coords)}')

# Visa alla områden med koordinater
print(f'\nExempel:')
for omrade, coord in list(area_coords.items())[:10]:
    dist = geodesic(coord, CENTRUM).km
    print(f'  {omrade:25s}: ({coord[0]:.4f}, {coord[1]:.4f}) — {dist:.1f} km från centrum')

  Radhus Lillån → fallback till centrum
  Radhus Ormesta → fallback till centrum
  Lägenhet Rynningeåsen → fallback till centrum
  Södra Lindhult → fallback till centrum
  Lägenhet Örebro → fallback till centrum
  Lägenhet Södra Ladugårdsängen → fallback till centrum
  Rynningeåsen → fallback till centrum
  Radhus Mosås → fallback till centrum
  Lägenhet Väster → fallback till centrum
Totalt geokodade: 40

Exempel:
  Örebro                   : (59.2747, 15.2151) — 0.1 km från centrum
  Adolfsberg               : (59.2390, 15.1835) — 4.4 km från centrum
  Almby                    : (59.2606, 15.2506) — 2.7 km från centrum
  Sörbyängen               : (59.2536, 15.2266) — 2.5 km från centrum
  Hovsta                   : (59.3467, 15.2321) — 8.0 km från centrum
  Mellringe                : (59.2980, 15.1625) — 3.8 km från centrum
  Lundby                   : (59.3089, 15.1843) — 4.1 km från centrum
  Sörby                    : (59.2603, 15.2252) — 1.8 km från centrum
  Solhaga            

In [12]:
# ============================================================
# STEG 6: Beräkna avstånd för varje bostad
# ============================================================

# Koppla koordinater till varje bostad via omrade_clean
df['latitude'] = df['omrade_clean'].map(lambda x: area_coords.get(x, CENTRUM)[0])
df['longitude'] = df['omrade_clean'].map(lambda x: area_coords.get(x, CENTRUM)[1])

# Beräkna avstånd till viktiga platser (i km)
def calc_distance(row, target):
    try:
        return geodesic((row['latitude'], row['longitude']), target).km
    except:
        return np.nan

df['avstand_centrum_km'] = df.apply(lambda r: calc_distance(r, CENTRUM), axis=1)
df['avstand_station_km'] = df.apply(lambda r: calc_distance(r, CENTRALSTATION), axis=1)
df['avstand_sjukhus_km'] = df.apply(lambda r: calc_distance(r, SJUKHUSET), axis=1)
df['avstand_universitet_km'] = df.apply(lambda r: calc_distance(r, UNIVERSITETET), axis=1)
df['avstand_marieberg_km'] = df.apply(lambda r: calc_distance(r, MARIEBERG), axis=1)

print('Avstånd beräknade!')
print(f'\navstand_centrum_km:')
print(f'  Min:    {df["avstand_centrum_km"].min():.1f} km')
print(f'  Median: {df["avstand_centrum_km"].median():.1f} km')
print(f'  Max:    {df["avstand_centrum_km"].max():.1f} km')

print(f'\nNärmast centrum: {df.loc[df["avstand_centrum_km"].idxmin(), "omrade_clean"]}')
print(f'Längst från centrum: {df.loc[df["avstand_centrum_km"].idxmax(), "omrade_clean"]}')

Avstånd beräknade!

avstand_centrum_km:
  Min:    0.0 km
  Median: 0.1 km
  Max:    21.8 km

Närmast centrum: Rynninge
Längst från centrum: Odensbacken


In [14]:
# ============================================================
# STEG 7: Korrelationscheck — påverkar avstånd priset?
# ============================================================

avstand_cols = ['avstand_centrum_km', 'avstand_station_km', 
                'avstand_sjukhus_km', 'avstand_universitet_km',
                'avstand_marieberg_km']

print('Korrelation med slutpris:')
print('=' * 55)
for col in avstand_cols:
    c = df[[col, 'slutpris']].dropna().corr().iloc[0, 1]
    bar = '█' * int(abs(c) * 40)
    sign = '+' if c > 0 else '-'
    print(f'  {col:30s}: {sign}{abs(c):.3f}  {bar}')

# Också latitude och longitude
for col in ['latitude', 'longitude']:
    c = df[[col, 'slutpris']].dropna().corr().iloc[0, 1]
    print(f'  {col:30s}: {"+" if c > 0 else "-"}{abs(c):.3f}')

Korrelation med slutpris:
  avstand_centrum_km            : -0.032  █
  avstand_station_km            : -0.029  █
  avstand_sjukhus_km            : -0.029  █
  avstand_universitet_km        : -0.095  ███
  avstand_marieberg_km          : -0.048  █
  latitude                      : -0.097
  longitude                     : +0.004


In [15]:
# ============================================================
# STEG 8: Spara den berikade datan med koordinater
# ============================================================

output_path = '../data/processed/orebro_housing_enriched.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'Sparad: {output_path}')
print(f'Rader: {len(df)}')
print(f'Kolumner: {len(df.columns)}')

print(f'\nNya kolumner:')
new = ['latitude', 'longitude', 'gatuadress',
       'avstand_centrum_km', 'avstand_station_km', 
       'avstand_sjukhus_km', 'avstand_universitet_km',
       'avstand_marieberg_km']
for col in new:
    if col in df.columns:
        print(f'  + {col}')

# Spara även koordinater separat (för kartan)
coords_df = pd.DataFrame([
    {'omrade': k, 'lat': v[0], 'lon': v[1]} 
    for k, v in area_coords.items()
])
coords_df.to_csv('../data/processed/area_coordinates.csv', index=False)
print(f'\nKoordinater sparade: {len(coords_df)} områden')

print(f'\n{"="*50}')
print(f'KLART!')
print(f'Lägg till avstand_centrum_km i modellens features')
print(f'Koordinaterna används sedan för kartan i dashboarden')
print(f'{"="*50}')

Sparad: ../data/processed/orebro_housing_enriched.csv
Rader: 6600
Kolumner: 49

Nya kolumner:
  + latitude
  + longitude
  + gatuadress
  + avstand_centrum_km
  + avstand_station_km
  + avstand_sjukhus_km
  + avstand_universitet_km
  + avstand_marieberg_km

Koordinater sparade: 40 områden

KLART!
Lägg till avstand_centrum_km i modellens features
Koordinaterna används sedan för kartan i dashboarden
